<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
M99 Skill Compliance
</b></font> </br></p>

---

In [42]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M99-Skill-Compliance"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.10
langchain-chroma                         1.1.0
langchain-classic                        1.0.2
langchain-community                      0.4.1
langchain-core                           1.2.17
langchain-ollama                         1.0.1
langchain-openai                         1.1.10
langchain-text-splitters                 1.1.1
langgraph                                1.0.10
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.9

IP-Adresse: 35.229.167.58
Hostname: 58.167.229.35.bc.googleusercontent.com
Stadt: Taipei
Region: Taiwan
Land: TW
Koordinaten: 25.0531,121.5264
Provider: AS396982 Google LLC
Postleitzahl: None
Zeitzone: Asia/Taipei


# 1 | Übersicht
---

Dieses Modul zeigt, warum **Skills** ein wichtiges Mittel zur Steuerung von Agenten sind.

Ein Skill ist kein normaler Prompt, sondern ein **wiederverwendbares Arbeitsrezept**:
- wann ein bestimmtes Vorgehen aktiviert wird
- welche Prüfschritte in welcher Reihenfolge nötig sind
- welche Regeln, Referenzen und Hilfsskripte der Agent heranzieht

Im Mittelpunkt steht hier ein **Compliance-Skill**. Er demonstriert drei Kernideen:

| Idee | Bedeutung im Agenten-Kontext |
|---|---|
| **Workflow-Orchestrierung** | Der Agent arbeitet feste Prüfschritte in Reihenfolge ab |
| **Guardrails** | Kritische Prüfungen werden nicht stillschweigend übersprungen |
| **Progressive Disclosure** | Nur die Kernlogik liegt in `SKILL.md`, Details werden bei Bedarf geladen |

**Lernziele:**
- Skills als Mittel zur Agentensteuerung einordnen
- eine gute Trennung zwischen `SKILL.md`, `references/` und `scripts/` verstehen
- einen Compliance-Skill lokal analysieren und testen


# 2 | Skill-Design für Agenten
---

<p><font color='black' size="5">Warum gehört das in einen Agenten-Kurs?</font></p>

Skills operationalisieren Fachwissen. Der Agent bekommt dadurch nicht nur Antworten, sondern eine **strukturierte Handlungslogik**.

Gerade bei Compliance-, Security- oder Freigabeprozessen reicht ein freier Prompt oft nicht aus. Man will stattdessen:
- Pflichtprüfungen definieren
- Eskalationsregeln festlegen
- Entscheidungsdokumentation erzwingen
- wiederkehrende Logik verlässlich wiederverwenden

<p><font color='black' size="5">Empfohlene Aufteilung</font></p>

| Datei-/Ordner-Typ | Rolle |
|---|---|
| `SKILL.md` | Kernablauf, Trigger, Guardrails, Verweise |
| `references/*.md` | Detailwissen, Regeln, Checklisten, Beispiele |
| `scripts/` | deterministische Hilfslogik für fragile oder wiederkehrende Schritte |

Das ist genau der Punkt, den Ihre ursprüngliche Frage adressiert hat: **Ja, Skills dürfen detailliert sein, aber die Details sollten sauber aufgeteilt werden.**

In [43]:
import urllib.request

GITHUB_BASE = "https://raw.githubusercontent.com/ralf-42/Agenten/main/06_skill/compliance"

def fetch_skill_file(path: str) -> str:
    url = f"{GITHUB_BASE}/{path}"
    with urllib.request.urlopen(url) as r:
        return r.read().decode("utf-8")

# Überblick über verfügbare Dateien im Skill
skill_files = [
    "SKILL.md",
    "references/checklist.md",
    "references/risk_rules.md",
    "references/examples.md",
    "scripts/assess_risk.py",
]
print(f"Skill-Quelle: {GITHUB_BASE}\n")
for f in skill_files:
    print(f"  {f}")

Skill-Quelle: https://raw.githubusercontent.com/ralf-42/Agenten/main/06_skill/compliance

  SKILL.md
  references/checklist.md
  references/risk_rules.md
  references/examples.md
  scripts/assess_risk.py


# 3 | Hands-On: Compliance-Skill lesen
---

In [44]:
skill_text = fetch_skill_file("SKILL.md")
print(skill_text[:3000])

---
name: compliance-skill
description: Use this skill when the user wants a compliance-oriented agent workflow, including sanctions checks, risk scoring, approval gates, documentation duties, or controlled multi-step onboarding and payment release processes.
---

# Compliance Skill

This skill defines a reusable compliance workflow for agentic processes with approval and documentation requirements.

## Quick Start

Use this skill when a request involves one or more of these patterns:

- A transaction, supplier, customer, or account must be checked before execution
- The agent must follow a fixed review sequence before taking action
- The user asks for sanctions checks, KYC/KYB checks, risk scoring, approval gates, or audit logging
- The task needs a documented go/no-go decision

If the request is purely informational and no operational decision is needed, give a normal answer and skip the full workflow.

## Core Workflow

Follow these steps in order unless the user explicitly narrows 

Achten Sie beim Lesen auf drei Dinge:
- **Trigger-Beschreibung:** Wann soll der Skill aktiv werden?
- **Core Workflow:** Welche Schritte sind verpflichtend?
- **Guardrails:** Was darf der Agent nicht behaupten oder überspringen?


In [45]:
for name in ['checklist.md', 'risk_rules.md', 'examples.md']:
    content = fetch_skill_file(f"references/{name}")
    print(f'\n### {name}\n')
    print(content[:1500])


### checklist.md

# Compliance Checklist

Use this checklist when the skill requires an operational decision.

## 1. Intake

- Identify the subject and case type.
- Capture country and transaction context.
- Verify whether the task is informational, advisory, or operational.

## 2. Identity And Context

- Confirm the party name.
- Confirm whether the party is an individual or legal entity.
- Record any known counterparties.
- Record the declared business purpose.

## 3. Required Screening

- Sanctions screening
- Geography screening
- Transaction size check
- Adverse indicator review
- Documentation completeness

## 4. Escalation Triggers

Escalate immediately if one of these is true:

- possible sanctions hit
- high-risk country
- unusually large transaction
- unclear business purpose
- inconsistent documents
- politically exposed person or equivalent high-risk marker

## 5. Documentation

The final note should contain:

- case summary
- checks performed
- observed indicators
- final

# 4 | Hands-On: deterministische Risiko-Prüfung
---

Der Skill enthält zusätzlich ein kleines Skript. Das ist nützlich, wenn ein Teil der Logik **nicht nur beschrieben, sondern zuverlässig berechnet** werden soll.

In [46]:
import json

# assess_risk.py aus GitHub laden — in eigenem Namespace ausführen,
# damit der if __name__ == "__main__" Block nicht getriggert wird
script_code = fetch_skill_file("scripts/assess_risk.py")
_ns = {"__name__": "__exec__"}
exec(script_code, _ns)
assess = _ns["assess"]  # Funktion in den Notebook-Scope holen

case = {
    'subject_type': 'vendor',
    'subject_name': 'Example Trading LLC',
    'country': 'RU',
    'transaction_amount': 18000,
    'sanctions_hit': False,
    'adverse_media_hit': True,
    'pep_flag': False,
    'documents_complete': True,
}

risk_result = assess(case)
print(json.dumps(risk_result, indent=2))

{
  "risk_score": 70,
  "risk_level": "high",
  "decision": "block",
  "reasons": [
    "adverse_media_hit",
    "high_risk_country:RU",
    "large_transaction"
  ]
}


In [47]:
from textwrap import dedent

decision_note = dedent(f'''
### Compliance Decision

- Case: {case['subject_type']} for {case['subject_name']}
- Checks performed: sanctions screening, geography review, transaction size review, documentation check
- Risk level: {risk_result['risk_level']}
- Decision: {risk_result['decision']}
- Rationale: {', '.join(risk_result['reasons'])}
- Missing information or escalation point: none in this training example
- Audit note: simulated training assessment, no live sanctions source connected
''').strip()

print(decision_note)

### Compliance Decision

- Case: vendor for Example Trading LLC
- Checks performed: sanctions screening, geography review, transaction size review, documentation check
- Risk level: high
- Decision: block
- Rationale: adverse_media_hit, high_risk_country:RU, large_transaction
- Missing information or escalation point: none in this training example
- Audit note: simulated training assessment, no live sanctions source connected


# 5 | Agent mit Compliance-Skill
---

Bisher haben wir den Skill manuell gelesen und das Skript direkt aufgerufen. Jetzt übergeben wir beides an einen Agenten:

- **`SKILL.md`** wird als System-Prompt geladen — der Agent kennt damit den vollständigen Compliance-Workflow
- **`assess_risk`** wird als `@tool` bereitgestellt — der Agent kann die deterministische Prüfung selbst aufrufen
- Der Agent entscheidet eigenständig, wann er das Tool einsetzt und wie er das Ergebnis kommuniziert

| Komponente | Rolle |
|---|---|
| System-Prompt (SKILL.md) | Steuert Verhalten, Pflichtschritte, Guardrails |
| `@tool assess_risk` | Deterministische Risikobewertung |
| Agent | Orchestriert beides, erzeugt Compliance Decision |

In [48]:
from langchain_core.tools import tool

@tool
def compliance_check(
    subject_type: str,
    subject_name: str,
    country: str,
    transaction_amount: float = 0,
    sanctions_hit: bool = False,
    adverse_media_hit: bool = False,
    pep_flag: bool = False,
    documents_complete: bool = True,
) -> str:
    """Führt eine deterministische Compliance-Risikoprüfung durch und gibt
    risk_score, risk_level, decision und reasons zurück."""
    case = {
        "subject_type": subject_type,
        "subject_name": subject_name,
        "country": country,
        "transaction_amount": transaction_amount,
        "sanctions_hit": sanctions_hit,
        "adverse_media_hit": adverse_media_hit,
        "pep_flag": pep_flag,
        "documents_complete": documents_complete,
    }
    return json.dumps(assess(case), indent=2)

print("Tool registriert:", compliance_check.name)

Tool registriert: compliance_check


In [49]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# SKILL.md als System-Prompt laden
skill_system_prompt = fetch_skill_file("SKILL.md")

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
memory = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[compliance_check],
    prompt=skill_system_prompt,
    checkpointer=memory,
)

# Thread-ID definiert die Konversationssession
session = {"configurable": {"thread_id": "compliance-demo-1"}}

print("Agent erstellt mit create_react_agent + MemorySaver")

Agent erstellt mit create_react_agent + MemorySaver


In [ ]:
import time

# Neue Session für jeden Testlauf (verhindert Überlappung mit vorherigen Runs)
session = {"configurable": {"thread_id": f"compliance-{int(time.time())}"}}

# ── Aufruf 1: vollständiger Fließtext — sanctions_clearance_confirmed fehlt ──
user_input_1 = (
    "Wir wollen einen neuen Lieferanten onboarden: "
    "Example Trading LLC aus Russland. "
    "Transaktionsvolumen 18.000 EUR. "
    "Geschäftszweck: Einkauf von Industriekomponenten. "
    "Zahlungskontext: reguläre Überweisung nach Wareneingang. "
    "Es gibt Hinweise auf negative Medienberichte. "
    "Alle Dokumente liegen vor."
)

response_1 = agent.invoke(
    {"messages": [("human", user_input_1)]},
    config=session,
)
mprint(response_1["messages"][-1].content)

⬆️ **Guardrail aktiv:** Alle anderen Felder extrahiert der Agent aus dem Fließtext. Nur `sanctions_clearance_confirmed` fragt er explizit nach — dieses Feld kann nicht inferiert werden und ist im SKILL.md als nicht-ableitbar markiert.

Der User bestätigt jetzt die Sanctions-Prüfung — der Agent hat den vollen Kontext durch das Memory:

In [ ]:
# ── Aufruf 2: User bestätigt Sanctions-Prüfung — gleiche session! ────────────
user_input_2 = "Die Sanctions-Prüfung wurde formal durchgeführt, kein Treffer."

response_2 = agent.invoke(
    {"messages": [("human", user_input_2)]},
    config=session,       # ← selbe session wie Aufruf 1
)
mprint(response_2["messages"][-1].content)

In [52]:
show_trace("M99-Skill-Compliance", show_steps=True)

## LangSmith Trace — `M99-Skill-Compliance`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `Prompt` | ✅ success | 0.0s | 0 |
| `RunnableSequence` | ❌ pending | — | 0 |
| `call_model` | ❌ pending | — | 0 |
| `agent` | ❌ pending | — | 0 |
| `LangGraph` | ❌ pending | — | 0 |

> Keine Child-Runs gefunden.

# 6 | Fazit
---

Dieses Beispiel zeigt die saubere Rollenverteilung eines Skills:
- `SKILL.md` steuert das Verhalten des Agenten
- `references/` halten Fachregeln und Beispiele aus dem Kernkontext heraus
- `scripts/` übernehmen deterministische Teilaufgaben

Genau deshalb passt das Thema in den Kurs **Agenten**: Es geht nicht nur um Antworten, sondern um **zuverlässig gesteuerte agentische Prozesse**.

# A | Aufgabe
---

<p><font color='black' size="5">
Skill erweitern
</font></p>

Erweitern Sie den Compliance-Skill um einen zweiten Entscheidungspfad, zum Beispiel für Lieferanten-Onboarding oder Auszahlungsfreigaben.

**Teilaufgaben:**
- Ergänzen Sie mindestens eine neue Regel in `references/risk_rules.md`
- Fügen Sie ein weiteres Beispiel in `references/examples.md` hinzu
- Passen Sie das Skript so an, dass die neue Regel in die Bewertung einfließt
